## Executive Summary

**Purpose:** Provide realistic material properties for motor design

**What it does:** Store and query electrical steel material data (B-H curves, core losses).

### Why It Matters
Motor design requires core loss calculations; lookup actual steel grade (M-19, M-27) properties.

### Quick Start
**Inputs:** Steel grade name, operating frequency, flux density

**Outputs:** B-H curve data, core loss at given conditions

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Feeds material properties to 04_material_models.ipynb and loss calculations

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import pandas as pd
from dataclasses import dataclass, field
from pathlib import Path
import functools
import pickle
import os
import numpy as np
from emachines.magnetics import SteelDatabase

## Theory: Electrical Steel

Silicon-steel optimized for low core loss.

**Properties:**
- Silicon content (0.5–3.5%) increases resistivity
- Reduces eddy current losses
- Laminations further reduce losses

**Manufacturers:**
- Voestalpine (Austria) — ISOVAC
- ThyssenKrupp (Germany) — PowerCore
- SURA (Sweden) — High-permeability

In [2]:
#| export
@dataclass
class SteelGrade:
    """Container for electrical steel grade properties."""
    name:         str
    manufacturer: str
    bh_data:      pd.DataFrame = field(default_factory=pd.DataFrame)
    loss_data:    pd.DataFrame = field(default_factory=pd.DataFrame)
    source_file:  str = ""
    
    @property
    def frequencies(self) -> list[float]:
        """Available frequencies in loss dataset (Hz)."""
        if self.loss_data.empty or "frequency [Hz]" not in self.loss_data.columns:
            return []
        return sorted(self.loss_data["frequency [Hz]"].unique().tolist())
    
    @property
    def bh_points(self) -> int:
        """Number of B-H data points."""
        return len(self.bh_data)
    
    def loss_at(self, freq: float, B: float) -> float:
        """Interpolate core loss at given frequency and flux density."""
        if freq not in self.frequencies:
            raise ValueError(f"Frequency {freq} Hz not available")
        b_col = next((c for c in ("B [T]", "J [T]") if c in self.loss_data.columns), None)
        loss_col = next((c for c in ("core loss P [W/kg]", "Core Loss [W/kg]") 
                         if c in self.loss_data.columns), None)
        if not b_col or not loss_col:
            raise ValueError("Loss data missing expected columns.")
        subset = self.loss_data[self.loss_data["frequency [Hz]"] == freq].sort_values(b_col)
        return float(np.interp(B, subset[b_col].values, subset[loss_col].values))

In [ ]:
#| export
# Note: The SteelDatabase class in the compiled module handles
# loading steel data from the literature folder.
# This section demonstrates how to use it.

def load_literature_database(literature_path: str | Path = None) -> 'SteelDatabase':
    """
    Create a SteelDatabase instance from the literature folder.
    
    The literature folder should have this structure:
    ```
    literature/datasheets/
    ├── Voelstapine Electrical Steel/     (Excel files)
    ├── SURA Electrical Steel/             (Excel files)  
    ├── Thyssenkrupp Electrical Steel/     (Excel files)
    └── steel_data_cache/                  (Pickle files - preprocessed)
        ├── sura_m470-50a.pkl
        ├── Simulation-data-isovac310-50A-voestalpine-*.pkl
        └── ...
    ```
    
    Parameters
    ----------
    literature_path : str or Path
        Path to the literature folder. If None, searches in:
        - ../KnowledgeBase/literature (relative to notebook)
        - ./KnowledgeBase/literature (relative to current dir)
        - Environment variable EMACHINES_LITERATURE
    
    Returns
    -------
    SteelDatabase
        Database instance with access to all 51+ steel grades
    
    Examples
    --------
    >>> db = load_literature_database()
    >>> db.grades  # List all available grades
    >>> db.manufacturers  # Group by manufacturer
    >>> steel = db.load('SURA M470-50A')
    >>> steel.loss_at(freq=400, B=1.5)  # Get core loss
    """
    import os
    
    if literature_path is None:
        # Search for literature folder
        candidates = [
            os.environ.get('EMACHINES_LITERATURE'),
            Path.cwd() / 'KnowledgeBase' / 'literature' / 'datasheets',
            Path.cwd() / '..' / 'KnowledgeBase' / 'literature' / 'datasheets',
        ]
        literature_path = None
        for candidate in candidates:
            if candidate and Path(candidate).exists():
                literature_path = candidate
                break
    
    literature_path = Path(literature_path) if literature_path else None
    
    if not literature_path or not literature_path.exists():
        raise FileNotFoundError(
            f"Literature datasheets directory not found at {literature_path}. "
            f"Expected structure: KnowledgeBase/literature/datasheets/"
        )
    
    return SteelDatabase(str(literature_path))

In [ ]:
#| export
# Load all real steel grades from literature
# This will be created on first import
_LITERATURE_STEELS: dict = {}  # Lazy-loaded on first access

def get_steel_library(reload: bool = False) -> dict[str, 'SteelGrade']:
    """Get dictionary of available steel grades from literature.
    
    Lazily loads steel grades from the KnowledgeBase/literature folder
    on first access.
    
    Parameters
    ----------
    reload : bool
        Force reload all steel data (default: False)
    
    Returns
    -------
    dict
        Dictionary mapping steel grade names to SteelGrade objects
    """
    global _LITERATURE_STEELS
    if reload or not _LITERATURE_STEELS:
        try:
            _LITERATURE_STEELS = load_literature_steels()
        except FileNotFoundError:
            print("Warning: Literature folder not found. Using sample data only.")
            _LITERATURE_STEELS = {}
    return _LITERATURE_STEELS

In [3]:
#| export
SAMPLE_BH: dict = {
    "M-19 Steel": {
        "H (A/m)": [0, 50, 100, 150, 200, 300, 400, 500, 1000, 1500, 2000, 3000, 4000, 5000],
        "B (T)":   [0, 0.6, 1.0, 1.2, 1.3, 1.4, 1.45, 1.5, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85],
    },
    "M-36 Steel": {
        "H (A/m)": [0, 40, 80, 120, 160, 240, 320, 400, 800, 1200, 1600, 2400, 3200, 4000],
        "B (T)":   [0, 0.5, 0.9, 1.1, 1.2, 1.3, 1.35, 1.4, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75],
    },
}

In [4]:
#| export
SAMPLE_LOSS: dict = {
    "M-19 Steel": pd.DataFrame({
        "Frequency (Hz)":  [60, 60, 60, 60, 400, 400, 400, 400],
        "Flux Density (T)":[0.5, 1.0, 1.5, 1.7, 0.5, 1.0, 1.5, 1.7],
        "Core Loss (W/kg)":[0.4, 1.0, 1.8, 2.5, 4.0, 12.0, 25.0, 35.0],
    }),
    "M-36 Steel": pd.DataFrame({
        "Frequency (Hz)":  [60, 60, 60, 60, 400, 400, 400, 400],
        "Flux Density (T)":[0.5, 1.0, 1.5, 1.7, 0.5, 1.0, 1.5, 1.7],
        "Core Loss (W/kg)":[0.5, 1.2, 2.2, 3.0, 5.0, 15.0, 30.0, 42.0],
    }),
}

## Test

Verify steel grade functionality.

In [5]:
# Test with sample data (fallback)
m19_bh = SAMPLE_BH["M-19 Steel"]
m19_loss = SAMPLE_LOSS["M-19 Steel"].copy()
m19_loss.columns = ["frequency [Hz]", "B [T]", "core loss P [W/kg]"]

m19 = SteelGrade(
    name="M-19 Steel",
    manufacturer="sample",
    bh_data=pd.DataFrame(m19_bh),
    loss_data=m19_loss,
)

assert m19.bh_points == 14
assert m19.frequencies == [60, 400]
print(f"✓ SteelGrade: {m19.name}")
print(f"  B-H points: {m19.bh_points}")
print(f"  Frequencies: {m19.frequencies} Hz")

loss = m19.loss_at(400, 1.0)
assert loss > 0
print(f"✓ loss_at(400 Hz, 1.0 T) = {loss:.2f} W/kg")

# Test with real literature data using SteelDatabase
print("\n" + "="*70)
print("LOADING REAL STEEL DATA FROM LITERATURE")
print("="*70)

try:
    db = load_literature_database()
    print(f"\n✓ Loaded SteelDatabase from literature folder")
    print(f"  Total grades available: {len(db.grades)}")
    print(f"\n  Manufacturers:")
    for mfr, grades in db.manufacturers.items():
        print(f"    • {mfr}: {len(grades)} grades")
        print(f"      Examples: {', '.join(grades[:2])}")
    
    # Load and test a real steel grade
    if db.grades:
        test_grade = db.grades[0]
        steel = db.load(test_grade)
        print(f"\n✓ Loaded test grade: {steel.name}")
        print(f"  Manufacturer: {steel.manufacturer}")
        print(f"  B-H curve points: {steel.bh_points}")
        print(f"  Available frequencies: {steel.frequencies} Hz")
        
        if steel.frequencies:
            loss = steel.loss_at(steel.frequencies[0], 1.0)
            print(f"  Core loss @ {steel.frequencies[0]} Hz, 1.0 T: {loss:.4f} W/kg")

except FileNotFoundError as e:
    print(f"\n⚠️  Literature folder not found (expected for demo)")
    print(f"    Set EMACHINES_LITERATURE or place data in KnowledgeBase/literature/datasheets")

print("\n✓ All tests passed!")

✓ SteelGrade: M-19 Steel
  B-H points: 14
  Frequencies: [60, 400] Hz
✓ loss_at(400 Hz, 1.0 T) = 12.00 W/kg

All tests passed!
